# 01 Train Backbones

This notebook trains the candidate CNN backbone models once and saves the best checkpoints.

Models:
- ResNet50
- EfficientNet-B0
- MobileNetV2

Outputs:
- best checkpoints
- training history CSV files
- training configuration JSON
- class mapping JSON
- training summary CSV
- ZIP archive of training outputs

In [11]:
# ----------------------------------------
# Section 1: Imports
# ----------------------------------------

import os
import json
import time
import random
import zipfile
from pathlib import Path
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [2]:
# ----------------------------------------
# Section 2: Reproducibility setup
# ----------------------------------------

SEED = 42

def seed_everything(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across Python, NumPy, and PyTorch.
    """
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id: int) -> None:
    """
    Ensure each DataLoader worker uses a deterministic seed.
    """
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

seed_everything(SEED)

print("Reproducibility setup completed")
print(f"Global seed: {SEED}")

Reproducibility setup completed
Global seed: 42


In [25]:
# ----------------------------------------
# Section 3: Configuration
# ----------------------------------------

CONFIG = {
    "seed": SEED,
    "image_size": 224,
    "batch_size": 32,
    "num_workers": 0,
    "epochs": 10,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "plantvillage_root": "/kaggle/input/datasets/thedataeng/plantvillage",
    "output_root": "/kaggle/working/thesis_outputs/train_backbones",
    "save_checkpoints": True,
    "mixed_precision": True,
}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_ROOT = Path(CONFIG["output_root"])
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

MODEL_NAMES = ["resnet50", "efficientnet_b0", "mobilenet_v2"]

print("Training configuration loaded")
print(f"Device: {DEVICE}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Models to train: {MODEL_NAMES}")

Training configuration loaded
Device: cuda
Output root: /kaggle/working/thesis_outputs/train_backbones
Models to train: ['resnet50', 'efficientnet_b0', 'mobilenet_v2']


In [26]:
# ----------------------------------------
# Section 4: Helper functions
# ----------------------------------------

def ensure_dir(path: Path) -> Path:
    """
    Create a directory if it does not exist and return the Path object.
    """
    path.mkdir(parents=True, exist_ok=True)
    return path

def get_train_transform(image_size: int = 224):
    """
    Create the training transform with basic augmentation.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def get_eval_transform(image_size: int = 224):
    """
    Create the evaluation transform.
    """
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

def count_parameters(model: nn.Module) -> int:
    """
    Count the number of trainable parameters in the model.
    """
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def checkpoint_path(model_name: str) -> Path:
    """
    Build the checkpoint path for a given model.
    """
    ckpt_dir = ensure_dir(OUTPUT_ROOT / "checkpoints")
    return ckpt_dir / f"{model_name}_seed{SEED}_best.pt"

def create_model(model_name: str, num_classes: int) -> nn.Module:
    """
    Create a pretrained backbone model and replace the final classification head.
    """
    if model_name == "resnet50":
        model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif model_name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    else:
        raise ValueError(f"Unsupported model name: {model_name}")

    return model.to(DEVICE)

@torch.no_grad()
def evaluate_model(model: nn.Module, loader: DataLoader) -> dict:
    """
    Evaluate the model on a validation loader and return basic classification metrics.
    """
    model.eval()

    all_preds = []
    all_labels = []
    total_loss = 0.0

    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model(images)
        loss = criterion(logits, labels)

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * images.size(0)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)

    accuracy = accuracy_score(all_labels, all_preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        all_labels,
        all_preds,
        average="macro",
        zero_division=0
    )

    return {
        "loss": total_loss / len(loader.dataset),
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
    }

print('Done')

Done


In [27]:
# ----------------------------------------
# Section 5: Dataset loading
# ----------------------------------------

pv_root = Path(CONFIG["plantvillage_root"])
train_dir = pv_root / "train"
val_dir = pv_root / "val"

assert train_dir.exists(), f"Missing PlantVillage train directory: {train_dir}"
assert val_dir.exists(), f"Missing PlantVillage val directory: {val_dir}"

train_tfms = get_train_transform(CONFIG["image_size"])
eval_tfms = get_eval_transform(CONFIG["image_size"])

train_dataset = datasets.ImageFolder(train_dir, transform=train_tfms)
val_dataset = datasets.ImageFolder(val_dir, transform=eval_tfms)

NUM_CLASSES = len(train_dataset.classes)

generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    worker_init_fn=seed_worker,
    generator=generator,
    pin_memory=True,
)

print("Datasets loaded successfully")
print(f"Training samples:   {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Number of classes:  {NUM_CLASSES}")

Datasets loaded successfully
Training samples:   25795
Validation samples: 5518
Number of classes:  27


In [28]:
# ----------------------------------------
# Section 6: Save training metadata
# ----------------------------------------

meta_dir = ensure_dir(OUTPUT_ROOT / "metadata")

config_path = meta_dir / "train_config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

class_map_path = meta_dir / "class_mapping.json"
with open(class_map_path, "w") as f:
    json.dump(
        {
            "classes": train_dataset.classes,
            "class_to_idx": train_dataset.class_to_idx,
        },
        f,
        indent=2
    )

print("Training metadata saved successfully")
print(f"Training config: {config_path}")
print(f"Class mapping:   {class_map_path}")

Training metadata saved successfully
Training config: /kaggle/working/thesis_outputs/train_backbones/metadata/train_config.json
Class mapping:   /kaggle/working/thesis_outputs/train_backbones/metadata/class_mapping.json


In [29]:
# ----------------------------------------
# Section 7: Training functions
# ----------------------------------------

def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scaler: torch.cuda.amp.GradScaler,
    criterion: nn.Module,
    epoch: int,
    total_epochs: int,
    model_name: str,
) -> float:
    """
    Train the model for one epoch and return the average training loss.
    """
    model.train()
    running_loss = 0.0

    progress_bar = tqdm(
        loader,
        desc=f"{model_name} | Epoch {epoch}/{total_epochs}",
        leave=False
    )

    for images, labels in progress_bar:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(CONFIG["mixed_precision"] and DEVICE.type == "cuda")):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        current_loss = running_loss / max(1, len(loader.dataset))
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader.dataset)

def train_model(model_name: str) -> pd.DataFrame:
    """
    Train a single backbone model and save the best checkpoint based on validation F1-score.
    Return a DataFrame containing the epoch-by-epoch training history.
    """
    print(f"Preparing model: {model_name}")

    model = create_model(model_name, NUM_CLASSES)
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=CONFIG["learning_rate"],
        weight_decay=CONFIG["weight_decay"]
    )
    scaler = torch.amp.GradScaler("cuda", enabled=(CONFIG["mixed_precision"] and DEVICE.type == "cuda"))
    criterion = nn.CrossEntropyLoss()

    history_rows = []
    best_val_f1 = -1.0
    best_epoch = -1
    ckpt_path = checkpoint_path(model_name)

    print(f"Checkpoint path: {ckpt_path}")
    print(f"Trainable parameters: {count_parameters(model):,}")

    for epoch in range(1, CONFIG["epochs"] + 1):
        print(f"Epoch {epoch}/{CONFIG['epochs']}")

        epoch_start = time.time()

        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            scaler=scaler,
            criterion=criterion,
            epoch=epoch,
            total_epochs=CONFIG["epochs"],
            model_name=model_name,
        )

        val_metrics = evaluate_model(model, val_loader)
        epoch_time = time.time() - epoch_start

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_f1": val_metrics["f1"],
            "epoch_time_sec": epoch_time,
        }
        history_rows.append(row)

        print(f"Training loss:      {train_loss:.4f}")
        print(f"Validation loss:    {val_metrics['loss']:.4f}")
        print(f"Validation accuracy:{val_metrics['accuracy']:.4f}")
        print(f"Validation F1-score:{val_metrics['f1']:.4f}")
        print(f"Epoch time (sec):   {epoch_time:.2f}")

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch

            if CONFIG["save_checkpoints"]:
                torch.save(model.state_dict(), ckpt_path)
                print("Best checkpoint updated")

        print("-" * 50)

    history_df = pd.DataFrame(history_rows)
    history_df["model_name"] = model_name
    history_df["best_epoch"] = best_epoch
    history_df["best_val_f1"] = best_val_f1

    print(f"Training completed for {model_name}")
    print(f"Best epoch: {best_epoch}")
    print(f"Best validation F1-score: {best_val_f1:.4f}")

    return history_df

print('Done')

Done


In [30]:
# ----------------------------------------
# Section 8: Train all backbone models
# ----------------------------------------

all_history = []

for idx, model_name in enumerate(MODEL_NAMES, start=1):
    print(f"Starting model {idx}/{len(MODEL_NAMES)}: {model_name}")
    history_df = train_model(model_name)
    all_history.append(history_df)
    print(f"Finished training model: {model_name}")
    print("=" * 60)

print("All backbone training runs completed successfully")

Starting model 1/3: resnet50
Preparing model: resnet50
Checkpoint path: /kaggle/working/thesis_outputs/train_backbones/checkpoints/resnet50_seed42_best.pt
Trainable parameters: 23,563,355
Epoch 1/10


resnet50 | Epoch 1/10:   0%|          | 0/807 [00:00<?, ?it/s]

Training loss:      0.3236
Validation loss:    0.0360
Validation accuracy:0.9882
Validation F1-score:0.9856
Epoch time (sec):   296.96
Best checkpoint updated
--------------------------------------------------
Epoch 2/10


resnet50 | Epoch 2/10:   0%|          | 0/807 [00:00<?, ?it/s]

Training loss:      0.0450
Validation loss:    0.0327
Validation accuracy:0.9899
Validation F1-score:0.9871
Epoch time (sec):   307.57
Best checkpoint updated
--------------------------------------------------
Epoch 3/10


...

Training loss:      0.0132
Validation loss:    0.0187
Validation accuracy:0.9933
Validation F1-score:0.9907
Epoch time (sec):   266.03
--------------------------------------------------
Epoch 9/10


mobilenet_v2 | Epoch 9/10:   0%|          | 0/807 [00:00<?, ?it/s]

Training loss:      0.0112
Validation loss:    0.0289
Validation accuracy:0.9926
Validation F1-score:0.9914
Epoch time (sec):   271.02
--------------------------------------------------
Epoch 10/10


mobilenet_v2 | Epoch 10/10:   0%|          | 0/807 [00:00<?, ?it/s]

Training loss:      0.0093
Validation loss:    0.0175
Validation accuracy:0.9957
Validation F1-score:0.9933
Epoch time (sec):   298.71
--------------------------------------------------
Training completed for mobilenet_v2
Best epoch: 6
Best validation F1-score: 0.9940
Finished training model: mobilenet_v2
All backbone training runs completed successfully


In [31]:
# ----------------------------------------
# Section 9: Save training histories
# ----------------------------------------

history_dir = ensure_dir(OUTPUT_ROOT / "histories")

combined_history = pd.concat(all_history, ignore_index=True)
combined_history_path = history_dir / "all_training_histories.csv"
combined_history.to_csv(combined_history_path, index=False)

for model_name in MODEL_NAMES:
    model_history = combined_history[combined_history["model_name"] == model_name].copy()
    model_history_path = history_dir / f"{model_name}_history.csv"
    model_history.to_csv(model_history_path, index=False)

print("Training histories saved successfully")
print(f"Combined history: {combined_history_path}")

Training histories saved successfully
Combined history: /kaggle/working/thesis_outputs/train_backbones/histories/all_training_histories.csv


In [32]:
# ----------------------------------------
# Section 10: Create training summary table
# ----------------------------------------

summary_rows = []

for model_name in MODEL_NAMES:
    model_history = combined_history[combined_history["model_name"] == model_name].copy()
    best_row = model_history.loc[model_history["val_f1"].idxmax()]

    temp_model = create_model(model_name, NUM_CLASSES)
    params = count_parameters(temp_model)

    summary_rows.append({
        "model_name": model_name,
        "best_epoch": int(best_row["epoch"]),
        "best_val_accuracy": float(best_row["val_accuracy"]),
        "best_val_precision": float(best_row["val_precision"]),
        "best_val_recall": float(best_row["val_recall"]),
        "best_val_f1": float(best_row["val_f1"]),
        "parameters": int(params),
        "checkpoint_path": str(checkpoint_path(model_name)),
    })

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_ROOT / "training_summary.csv"
summary_df.to_csv(summary_path, index=False)

print("Training summary saved successfully")
display(summary_df)

Training summary saved successfully


,model_name,best_epoch,best_val_accuracy,best_val_precision,best_val_recall,best_val_f1,parameters,checkpoint_path
0,resnet50,10,0.995651,0.994623,0.994820,0.994709,23563355,/kaggle/working/thesis_outputs/train_backbones...
1,efficientnet_b0,10,0.997463,0.996627,0.995546,0.996061,4042135,/kaggle/working/thesis_outputs/train_backbones...
2,mobilenet_v2,6,0.995832,0.994350,0.993717,0.994012,2258459,/kaggle/working/thesis_outputs/train_backbones...


In [33]:
# ----------------------------------------
# Section 11: Create ZIP archive
# ----------------------------------------

zip_path = OUTPUT_ROOT.parent / "01_train_backbones_outputs.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_ROOT.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=file_path.relative_to(OUTPUT_ROOT))

print("ZIP archive created successfully")
print(f"ZIP file: {zip_path}")
print("01_train_backbones notebook completed successfully")

ZIP archive created successfully
ZIP file: /kaggle/working/thesis_outputs/01_train_backbones_outputs.zip
01_train_backbones notebook completed successfully
